# Research Agent API - Usage Examples

This notebook demonstrates how to use the **ResearchClient** to get research responses with citations in Bigdata.com format.

## Features
- Simple synchronous interface
- Citations in standard Bigdata.com format
- Easy access to answer, citations, or both


## Setup


In [ ]:
import os
import sys
import json
import logging
from IPython.display import display, Markdown, JSON

# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Configure logging for research_client module
# (basicConfig doesn't work well in Jupyter, so we configure the logger directly)
logger = logging.getLogger("research_client")
logger.setLevel(logging.INFO)

# Clear any existing handlers to avoid duplicates
logger.handlers.clear()

# Create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# Add file handler (writes to output folder)
file_handler = logging.FileHandler("output/research_client.log", mode='w')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Prevent logs from propagating to root logger (which prints to console)
logger.propagate = False

# Import the client
from research_client import ResearchClient

print("✅ Client imported successfully!")
print("✅ Logging configured (INFO level) → output/research_client.log")


In [ ]:
# Create client (reads BIGDATA_API_KEY from environment)
# os.environ["BIGDATA_API_KEY"] = "your-api-key-here"

client = ResearchClient()
print("✅ Client ready")


## Execute Research Query


In [ ]:
# Execute research
print("🔍 Researching: 'What are the key risks Google is facing?'")
print("   This may take few seconds...\n")

query_message = """ What are the key risks Google is facing? """
#query_message = """ Generate a comprehensive daily macroeconomic morning briefing report for the US market. """
result = client.research(
    message=query_message,
    research_effort=  "lite" # "lite" OR "standard"
)

print(f"✅ Research complete!")
print(f"   Processing time: {result.processing_time_ms}ms")
print(f"   Citations found: {len(result.citations)}")


---
## A. Just Response

Display only the research answer (Markdown rendered):


In [ ]:
# Get just the answer
answer = result.get_answer()

display(Markdown(answer))


---
## B. Just Citations

Display only the citations in Bigdata.com format (JSON):


In [ ]:
# Get just the citations as JSON
citations = result.get_citations()

print(f"📚 Citations ({len(citations)} sources):\n")
print(json.dumps(citations, indent=2))


---
## C. Response with Citations

Display both answer and citations together:


In [ ]:
# Get full result as JSON (answer + citations)
full_result = result.to_dict()

print(json.dumps(full_result, indent=2))


### Formatted View (Answer + Citations)


In [ ]:
# Display answer as Markdown
display(Markdown("## Answer\n" + result.answer))

# Display citations in a readable format
display(Markdown("---\n## Citations"))

for i, citation in enumerate(result.citations[:10], 1):  # Show first 10
    c = citation.to_dict()
    
    # Build citation display
    parts = [f"### [{i}] {c.get('headline', 'N/A')}"]
    
    if c.get('source'):
        src = c['source']
        source_parts = []
        if src.get('name'):
            source_parts.append(f"**Source:** {src['name']}")
        if src.get('rank'):
            source_parts.append(f"**Rank:** {src['rank']}")
        if source_parts:
            parts.append(" | ".join(source_parts))
    
    if c.get('timestamp'):
        parts.append(f"**Date:** {c['timestamp']}")
    
    if c.get('url'):
        parts.append(f"**URL:** {c['url']}")
    
    # Show chunks
    if c.get('chunks'):
        parts.append("\n**Excerpts:**")
        for chunk in c['chunks']:
            text = chunk.get('text', '')
            if text:
                # Truncate long text
                display_text = text[:400] + "..." if len(text) > 400 else text
                display_text = display_text.replace('\n', ' ')
                parts.append(f"- *{display_text}*")
    
    display(Markdown("\n".join(parts) + "\n\n---"))

if len(result.citations) > 10:
    print(f"\n... and {len(result.citations) - 10} more citations")


---
## Save Results to File


In [ ]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Save just citations
with open("output/citations.json", "w") as f:
    f.write(result.get_citations_json())
print("✅ Saved: output/citations.json")

# Save full result (answer + citations)
with open("output/research_result.json", "w") as f:
    f.write(result.to_json())
print("✅ Saved: output/research_result.json")


---
## Citation Format Reference

The citations follow the standard Bigdata.com format:

```json
{
  "id": "E91DED180158906A74444B7837742178",
  "headline": "Article Title",
  "timestamp": "2026-01-06T15:00:30",
  "source": {
    "id": "5A5702",
    "name": "Benzinga",
    "rank": "RANK_1"
  },
  "url": "https://...",
  "chunks": [
    {
      "cnum": 5,
      "text": "Relevant text excerpt...",
      "relevance": 0.94,
      "sentiment": 0.82
    }
  ]
}
```

**Fields** (only non-null values are included):
- `id`: Document identifier
- `headline`: Article title
- `timestamp`: Publication date/time
- `source.id`: Source identifier
- `source.name`: Source name (e.g., "Benzinga", "Yahoo! Finance")
- `source.rank`: Source quality rank (e.g., "RANK_1")
- `url`: Document URL
- `chunks`: Array of relevant text excerpts with relevance scores


---
## D. Answer with Inline Citation Numbers

Display the answer with inline citation markers [1], [2], etc. and a numbered references section (like the screenshot):


In [ ]:
# Get answer with inline citation numbers
answer_with_citations = result.get_answer_with_citations()

# Get numbered citations that match the inline numbers
numbered_citations = result.get_numbered_citations()

print(f"📊 Found {len(numbered_citations)} inline citations\n")


In [ ]:
# Display answer with inline citation numbers [1], [2], etc.
display(Markdown("## Answer\n\n" + answer_with_citations))


In [ ]:
# Display numbered references section
display(Markdown("---\n## References\n"))

for citation in numbered_citations:
    num = citation.get('number', '?')
    headline = citation.get('headline', 'N/A')
    
    # Build citation card
    parts = [f"**[{num}]** {headline}"]
    
    # Source info
    source = citation.get('source', {})
    source_name = source.get('name') if source else citation.get('source_name')
    if source_name:
        parts.append(f"📰 **{source_name}**")
    
    # Date
    timestamp = citation.get('timestamp')
    if timestamp:
        parts.append(f"📅 {timestamp[:10]}")
    
    # URL
    url = citation.get('url')
    if url:
        parts.append(f"🔗 [{url[:50]}...]({url})")
    
    # Chunks/excerpts
    chunks = citation.get('chunks', [])
    if chunks:
        parts.append("\n**Excerpts:**")
        for chunk in chunks[:2]:  # Show max 2 excerpts
            text = chunk.get('text', '')
            if text:
                display_text = text[:300] + "..." if len(text) > 300 else text
                display_text = display_text.replace('\n', ' ')
                parts.append(f"- *{display_text}*")
    
    display(Markdown("\n".join(parts) + "\n\n---"))


### JSON Export with Inline Citations


In [ ]:
# Export as JSON with inline citations in the answer
result_with_inline = result.to_dict_with_inline_citations()

print(json.dumps(result_with_inline, indent=2)[:3000] + "\n... (truncated)")


In [ ]:
# Save result with inline citations
with open("output/result_with_inline_citations.json", "w") as f:
    f.write(result.to_json_with_inline_citations())
print("✅ Saved: output/result_with_inline_citations.json")
